# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nivyanaboruah/FlyRank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring**

My goal is to rank content pages by how urgently they should be reviewed. The model would assign a priority score to each page so that a content editor can review the most important pages first.

This is useful because not every page needs attention at the same time. Ranking helps prioritize limited review time. The score is decision-support and does not guarantee that a page will decline.

In [1]:
task_type = "ranking/scoring"
decision = "Which content page should be reviewed first?"

print("ML task:", task_type)
print("Decision:", decision)


ML task: ranking/scoring
Decision: Which content page should be reviewed first?


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: Whether a content page is declining.**

Ideally, I would predict whether a page's performance declines in a future measurement period. However, the current dataset uses `is_declining_label`, which is derived from `trend_direction` and `trend_pct`. Therefore, it is a rule-derived proxy rather than an independently observed future outcome. Any results using this label should be treated as preliminary and not as proof of future performance.

In [2]:
target_column = "is_declining_label"
target_type = "rule-derived proxy"

print("Target:", target_column)
print("Target type:", target_type)

Target: is_declining_label
Target type: rule-derived proxy


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: Precision@50**

Precision@50 measures the proportion of the 50 highest-ranked pages that are labelled as declining. A higher value means more of the selected pages match the target label. I will compare it with the overall proportion of declining labels as a baseline. Since the label is rule-derived, this metric measures agreement with the proxy rather than proven future predictive performance.

In [3]:
metric = "Precision@50"
k = 50

print("Success metric:", metric)
print("Ranking cutoff:", k)
print("Baseline: overall proportion of positive labels")


Success metric: Precision@50
Ranking cutoff: 50
Baseline: overall proportion of positive labels


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one content page.**

Each row represents one anonymized content page with its observed characteristics and performance metrics. I will inspect relevant features such as content age, days since last update, impressions, average position, click-through rate, and word count. The dataset's trend label will be treated as a proxy rather than an independent future outcome.

In [6]:
import os
import subprocess
from pathlib import Path
import pandas as pd

# Clone the FlyRank repository if it isn't already available
repo_dir = Path("/content/FlyRank")

if not repo_dir.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/nivyanaboruah/FlyRank.git",
            str(repo_dir)
        ],
        check=True
    )

# Find the dataset in the repository
matches = list(repo_dir.rglob("content_refresh_anonymized.csv"))

if not matches:
    raise FileNotFoundError(
        "CSV not found in the repository. "
        "Check the repository's data folder."
    )

data_path = matches[0]
df = pd.read_csv(data_path)

# Select columns relevant to this ML task
columns = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "trend_direction"
]

missing = [col for col in columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

lane_df = df[columns].copy()

print("Dataset loaded successfully!")
print("Dataset path:", data_path)
print("Dataset shape:", df.shape)
print("One row = one anonymized content page")

display(lane_df.head())

Dataset loaded successfully!
Dataset path: /content/FlyRank/data/raw/content_refresh_anonymized.csv
Dataset shape: (30000, 44)
One row = one anonymized content page


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,trend_direction
0,187,20,3803,10.6,0.76,3221.0,down
1,445,25,15320,20.3,0.05,2481.0,down
2,141,20,12581,36.5,0.09,3515.0,down
3,463,22,11751,6.2,0.49,NaN,stable
4,263,14,19140,44.0,0.13,2803.0,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why Machine Learning May Help

A fixed rule can prioritize pages using simple conditions, such as whether a page has not been updated recently and still receives many impressions. However, page performance may depend on several interacting factors, including content age, search visibility, average position, click-through rate, and word count.

A machine learning model may identify combinations of these signals that a simple rule misses. I would compare the model with the fixed-rule baseline using the same evaluation metric and an appropriate validation split.

However, ML is not automatically better. The current target is rule-derived, so any improvement would show agreement with that proxy rather than prove that the model can predict future page declines.

In [7]:
# Check how the available signals vary across trend categories

signals = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

print("Median signals by trend direction:")

display(
    df.groupby("trend_direction")[signals]
      .median()
      .round(2)
)

print("\nDataset size:", df.shape)
print("Distinct trend categories:",
      df["trend_direction"].nunique())


Median signals by trend direction:


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count
trend_direction,,,,,,
down,216.0,20.0,961.0,11.3,0.08,2909.0
flat,231.0,20.0,4.0,6.6,0.00,2698.5
new,279.0,20.0,3.0,0.0,0.00,2239.0
stable,300.0,22.0,1944.5,11.4,0.17,2912.5
up,291.5,22.0,587.0,15.3,0.09,2847.5



Dataset size: (30000, 44)
Distinct trend categories: 5


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.